<table><tr><th><div style='padding:15px;color:#030aa7;font-size:240%;text-align:center;font-weight:bold;font-family:Georgia,serif'>Consolidation des données météo</div></th></tr></table>

<div style='text-align:center'><img src='https://raw.githubusercontent.com/rbizoi/dockerBigDataBI/refs/heads/master/images/meteoFrance.png' width='1024'></div>

In [1]:
import pandas as pd, os, time

In [2]:
def tempsExecution(func):
    def chronometrerExécution(*_args, **_kwargs): 
        t0 = time.perf_counter()
        result = func(*_args, **_kwargs) 
        elapsed = time.perf_counter() - t0
        name = func.__name__
        args = ', '.join(repr(arg) for arg in _args) \
               if len(_args) == 1 \
               else ', '.join(arg+'='+repr(_kwargs[arg]) for arg in _kwargs)
        print(f'exécution {elapsed:0.8f}s\nfonction {name}({args})')
        return result
    return chronometrerExécution

In [4]:
@tempsExecution
def lectureFichiersMeteo(repertoire='../donnees/meteo'):
    meteo = pd.concat([ pd.read_csv(fichier,
                           sep=';',
                           usecols=['numer_sta','date','dd','ff','t','u','vv',
                                    'pres','rr1','rr3','rr6','rr12','rr24'],
                           na_values='mq',
                           dtype={'numer_sta':str,'date':str},
                          ).rename( columns={'numer_sta':'Station',
                                              'date':'DateHeure',
                                              'dd':'DirectionVent',
                                              'ff':'VitesseVent',
                                              't':'Temperature',
                                              'u':'Humidite',
                                              'vv':'Visibilite',
                                              'pres':'Pression',
                                              'rr1' :'Precipitation01',
                                              'rr3' :'Precipitation03',
                                              'rr6' :'Precipitation06',
                                              'rr12':'Precipitation12',
                                              'rr24':'Precipitation24'}) 
                         for fichier in [ f'{dirname}{os.sep}{filename}' for dirname, dirnames, filenames in os.walk(repertoire) 
                                                                         for filename in filenames]],
                        ignore_index=True)
    # meteo.reset_index().drop(columns='index',inplace=True)
    meteo["DateHeure"] = pd.to_datetime(meteo["DateHeure"], format='%Y%m%d%H%M%S')
    meteo.Temperature  = meteo.Temperature - 273.15
    meteo.Pression     = meteo.Pression / 100
    meteo.Visibilite   = meteo.Visibilite / 1000
    meteo['Precipitation'] =  meteo['Precipitation03'].combine_first(meteo['Precipitation06']/2)\
                                                      .combine_first(meteo['Precipitation12']/4)\
                                                      .combine_first(meteo['Precipitation24']/8)\
                                                      .combine_first(meteo['Precipitation01']*3)
    meteo.drop(columns=['Precipitation06',
                        'Precipitation12',
                        'Precipitation24',
                        'Precipitation01',
                        'Precipitation03'], inplace=True)
    meteo['Heure']        = meteo.DateHeure.dt.hour
    meteo['Jour']         = meteo.DateHeure.dt.dayofyear
    meteo['Mois']         = meteo.DateHeure.dt.month
    meteo['Annee']        = meteo.DateHeure.dt.year
    meteo['Semaine']      = meteo.DateHeure.dt.isocalendar().week
    meteo['MoisJour']     = meteo.DateHeure.dt.month*100 + meteo.DateHeure.dt.day
    meteo['AnneeMois']    = meteo.DateHeure.dt.year*100 + meteo.DateHeure.dt.month
    meteo['AnneeSemaine'] = meteo.DateHeure.dt.year*100 + meteo.DateHeure.dt.isocalendar().week
    meteo['AnneeJour']    = meteo.DateHeure.dt.year*1000 + meteo.DateHeure.dt.day
    meteo['JourNuit']     = meteo['Heure'].apply(lambda x: 'jour' if ( x >=7 ) & ( x<=18) else 'nuit')
    return meteo
    

In [5]:
meteo = lectureFichiersMeteo(repertoire='../donnees/meteo')

exécution 16.75036640s
fonction lectureFichiersMeteo(repertoire='../donnees/meteo')


In [7]:
postes = pd.read_csv('../donnees/postesSynop.csv',sep=';',dtype={'ID':str}).rename(columns={'ID':'Station'})
postes.Nom =postes.Nom.apply(lambda x : x if x in ['CLERMONT-FD','MONT-DE-MARSAN',
                                       'ST-PIERRE','ST-BARTHELEMY METEO'] 
                               else x[0:x.find('-')] 
                                    if x.find('-') != -1 else x).apply(lambda x : str(x).title())
postes.Altitude = postes.Altitude.astype('int16')
postes = postes[postes.Station < '08000']
postes.loc[postes.Latitude  < postes.Latitude.mean(),'Zone'] = 'S'
postes.loc[postes.Latitude  > postes.Latitude.mean(),'Zone'] = 'N'
postes.loc[postes.Longitude < postes.Longitude.mean(),'Zone'] += 'O'
postes.loc[postes.Longitude > postes.Longitude.mean(),'Zone'] += 'E'

In [8]:
meteo = postes.set_index('Station').join(meteo.set_index('Station')).reset_index()

In [9]:
meteo.head()

,Station,Nom,Latitude,Longitude,Altitude,Zone,DateHeure,DirectionVent,VitesseVent,Temperature,...,Heure,Jour,Mois,Annee,Semaine,MoisJour,AnneeMois,AnneeSemaine,AnneeJour,JourNuit
0,07005,Abbeville,50.136,1.834,69,NO,1996-01-01 00:00:00,170.0,2.0,2.89,...,0,1,1,1996,1,101,199601,199601,1996001,nuit
1,07005,Abbeville,50.136,1.834,69,NO,1996-01-01 03:00:00,130.0,3.0,2.70,...,3,1,1,1996,1,101,199601,199601,1996001,nuit
2,07005,Abbeville,50.136,1.834,69,NO,1996-01-01 06:00:00,120.0,3.0,2.89,...,6,1,1,1996,1,101,199601,199601,1996001,nuit
3,07005,Abbeville,50.136,1.834,69,NO,1996-01-01 09:00:00,130.0,4.1,3.60,...,9,1,1,1996,1,101,199601,199601,1996001,jour
4,07005,Abbeville,50.136,1.834,69,NO,1996-01-01 12:00:00,100.0,5.1,4.39,...,12,1,1,1996,1,101,199601,199601,1996001,jour


In [11]:
meteo = meteo[['Station', 'DateHeure', 'Nom', 'Latitude', 'Longitude', 'Altitude', 'Zone',
               'DirectionVent', 'VitesseVent', 'Temperature', 'Humidite',
               'Visibilite', 'Pression', 'Precipitation', 'Heure', 'Jour', 'Mois',
               'Annee', 'Semaine', 'MoisJour', 'AnneeMois', 'AnneeSemaine',
               'AnneeJour', 'JourNuit']]

In [12]:
meteo.to_parquet('../donnees/meteo.all.gzip',compression='gzip', engine='pyarrow')

CPU times: total: 0 ns
Wall time: 0 ns
